# **vae-1d-pixelwise — training**

Self-contained training notebook for **vae-1d-pixelwise** across the 4 datasets (IIRS / M3 / AVIRIS / CRIMS), trained under both loss regimes (standard = MSE+beta*KLD, physics = +lambda*SAM), saved to `model/<DATASET>/<model>_<loss>.pt`.

**Runs standalone — nothing from `utils/` is needed.** The config cell below inlines `utils/config.py`, `utils/hyperparams.py` and all four `hyperparam-config-*.yaml` files.

### Setup on Kaggle
1. Build the packed dataset on the lab machine and upload the resulting `dataset.zip` (see `scripts/train_fixed.sh`).
2. Add it to the notebook, then set `KAGGLE_INPUT` in the config cell to the dataset's mount point.
3. Enable the **GPU T4 x2** accelerator — the driver cell turns on `nn.DataParallel` automatically when it sees more than one device. Batch size is per dataset (IIRS 32, M3 32, AVIRIS 16, CRIMS 16) and gets split evenly across the GPUs.

### Two things to know
- **T4 is SM 7.5, so there is no bfloat16.** The loop falls back to float16 + `GradScaler`, while the lab and HPC (Ampere+) use bfloat16. If final reported numbers come from Kaggle, that dtype difference belongs in the paper's setup section.
- **The config cell duplicates repo constants.** Any change to band counts, model widths, or the spectral/spatial knobs in `utils/config.py` must be mirrored into all four notebooks.

## **Config**

In [ ]:
# =============================================================================
# CONFIG — fully self-contained (no `utils/` upload needed on Kaggle)
# =============================================================================
# Everything this notebook needs is in THIS cell: dataset band counts, the
# Settings dataclass, and the per-dataset hyper-parameters that live in
# utils/hyperparam_configs/*.yaml in the repo. Nothing is read from disk.
#
# !! DUPLICATION HAZARD !!
# This cell mirrors utils/config.py + utils/hyperparams.py + the four YAMLs.
# A change to any of those must be copied into ALL FOUR notebooks/*.ipynb.
# This is exactly how `CRIMS: 544` survived here after being wrong in the repo:
# the constant lived in five places. If you touch band counts, widths, or the
# spectral/spatial knobs, grep the notebooks too.
from dataclasses import dataclass, field

# --------------------------------------------------------------------------
# Datasets. CRIMS patches are 457 bands on disk; 457 is prime, so the spectral
# encoder's two stride-2 halvings can never be undone exactly (457//4 -> 114 ->
# 456 != 457). pack.py crops to 456, exactly as M3 is cropped 85 -> 84.
# --------------------------------------------------------------------------
DATASETS = {
    "IIRS":   {"input_channels": 256},
    "M3":     {"input_channels": 84},
    "AVIRIS": {"input_channels": 424},
    "CRIMS":  {"input_channels": 456},
}

# --------------------------------------------------------------------------
# Where the PACKED fp16 shards live: <root>/<split>.npy + <split>.json, built by
# utils/dataset/pack.py and shipped as dataset.zip (see scripts/train_fixed.sh).
# On Kaggle, add the dataset and set KAGGLE_INPUT to its mount point.
# --------------------------------------------------------------------------
import os as _os

KAGGLE_INPUT = "/kaggle/input/prism-hsi-packed"   # <-- edit to your dataset slug
_PACKED_BASE = (f"{KAGGLE_INPUT}/data/packed"
                if _os.path.isdir(KAGGLE_INPUT) else "data/packed")

DATA_ROOTS = {ds: f"{_PACKED_BASE}/{ds}" for ds in DATASETS}

# Kaggle's working dir is the only writable location.
CKPT_ROOT = "/kaggle/working/model" if _os.path.isdir("/kaggle/working") else "model"


@dataclass
class Settings:
    input_height: int = 64
    input_width: int = 64
    input_channels: int = 256          # overridden per dataset via make_settings()

    # training
    batch_size: int = 32
    num_workers: int = 4               # Kaggle gives 2-4 usable CPU cores
    epochs: int = 30
    lr: float = 1e-3
    beta: float = 1e-3
    lambda_physics: float = 0.3

    # spatial branch
    reduced_dims: int = 32
    latent_dim: int = 256
    n_2D_conv_blocks: int = 4
    conv2D_kernel_size: int = 3
    conv_output_c: int = field(init=False)
    conv_output_h: int = field(init=False)
    conv_output_w: int = field(init=False)

    # spectral branch
    spectral_n_1D_conv_blocks: int = 2
    spectral_conv1D_kernel_size: int = 4
    spectral_latent_dim: int = 4
    # Conv width of the spectral branch. Mirrors `reduced_dims` on the spatial
    # side. This used to be tied to input_channels, which made the branch cost
    # O(C^2) per pixel spectrum — 99.7% of vae-our's FLOPs — and swung vae-our's
    # parameter count 3x across sensors. Sequence length still tracks C, so the
    # latent stays sensor-aware; only the width is now a free hyper-parameter.
    spectral_base_ch: int = 32
    spectral_linear_expansion_dim: int = field(init=False)
    spectral_transpose_c: int = field(init=False)
    spectral_transpose_l: int = field(init=False)

    # Baseline capacity knobs — re-solved so each baseline matches vae-our's
    # parameter count at each dataset. Regenerate with
    # `python utils/check-model-params.py --solve`. IIRS defaults shown.
    vae_standard_base_ch: int = 86
    vae_standard_n_down: int = 3
    vae_standard_latent_ch: int = 256

    vae_3d_base_ch: int = 45
    vae_3d_n_down: int = 3
    vae_3d_latent_ch: int = 8

    vae_1d_hidden_dims: tuple = (2748, 1374, 687)
    vae_1d_latent_dim: int = 4

    def __post_init__(self):
        self.conv_output_c = self.reduced_dims * (2 ** self.n_2D_conv_blocks)
        self.conv_output_h = self.input_height // (2 ** self.n_2D_conv_blocks)
        self.conv_output_w = self.input_width // (2 ** self.n_2D_conv_blocks)
        self.spectral_transpose_c = self.spectral_base_ch * (2 ** (self.spectral_n_1D_conv_blocks - 1))
        self.spectral_transpose_l = self.input_channels // (2 ** self.spectral_n_1D_conv_blocks)
        self.spectral_linear_expansion_dim = self.spectral_transpose_c * self.spectral_transpose_l


# --------------------------------------------------------------------------
# Per-dataset hyper-parameters — INLINED from utils/hyperparam_configs/*.yaml
# so the notebook needs no repo files at all.
#
# batch_size is held constant across all 4 models x 2 loss regimes WITHIN a
# dataset — that is the axis the ablation compares on. It is NOT constant across
# datasets: there is no controlled comparison between sensors to protect, and
# memory per sample varies ~2x between them, so one global value would idle the
# GPU on the lighter sensors. It DOES hold across platforms for a given dataset,
# so each number is derived against the tightest budget (the lab's 24 GB) and
# reused unchanged here. Kaggle's 2xT4 sees batch_size/2 per device under
# nn.DataParallel, which clears in every case.
# Re-derive: PYTHONPATH=. python utils/find_max_batch.py --budget-gb 24 --fit
# --------------------------------------------------------------------------
# ------------------------------------------------------------------------------
# TWO independent controls are enforced below, and both matter:
#
#   LATENT RATE  — the information bottleneck, matched to 64:1 on every model.
#                  Reconstruction quality is bounded by rate almost by
#                  definition, so leaving it free (it used to vary 512x) makes
#                  the comparison meaningless. 64:1 is vae-3d's own natural
#                  point, so that baseline is unchanged, and it sits near Stable
#                  Diffusion's AutoencoderKL (48:1) — a credible LDM latent.
#   PARAMETERS   — the capacity of the function class, matched to vae-our.
#
# Latent SHAPE is deliberately NOT equalised: vae-standard is a spatial grid
# with no spectral axis, vae-our is a global vector beside a full-resolution
# per-pixel spectral map. That geometry IS the architecture under test.
#
# Regenerate (rate FIRST — it moves the parameter target):
#   python utils/match_latent_rate.py --exact
#   python utils/check-model-params.py --solve
#   python utils/find_max_batch.py --budget-gb 20 --fit    (scripts)
#   python utils/find_max_batch.py --budget-gb 13.5 --fit  (this notebook)
# ------------------------------------------------------------------------------
# NOTEBOOK BATCH TIER — sized for Kaggle's 2x T4 (30 GB total, ~13.5 GB usable
# per device after headroom). nn.DataParallel splits the batch, so each device
# carries batch_size/2.
#
# THIS IS NOT THE TIER THE SCRIPTS USE. utils/hyperparam_configs/*.yaml is sized
# for the lab's single 20 GB budget (32/32/16/16). A dataset trained partly here
# and partly there therefore carries TWO batch sizes, which is a within-dataset
# confound. It is recorded in every checkpoint's meta and every results row and
# flagged in VERDICT.txt. To avoid it entirely, give a whole dataset to one
# platform rather than splitting its cells across both.
#
# Per-device load here is 24/24/12/12, i.e. ~12 GB of a 15 GB T4 at the measured
# ~0.5 GB/sample (IIRS/M3) and ~0.9 GB/sample (AVIRIS/CRIMS). CONFIRM on the T4
# before a long run:
#     PYTHONPATH=. python utils/find_max_batch.py --budget-gb 13.5 --fit
# If a cell OOMs on Kaggle, the known-safe fallback is the script tier
# (32/32/16/16), which measures ~7.9 GB per device worst case.
BATCH_SIZE = {ds: b for ds, b in {
    "IIRS":   48,   # 24/device; binding: vae-3d, ~0.48 GB/sample
    "M3":     48,   # 24/device; binding: vae-1d, ~0.48 GB/sample (its MLP width
                    #            barely moves with band count, so it caps M3
                    #            before the 3D model does)
    "AVIRIS": 24,   # 12/device; binding: vae-3d, ~0.80 GB/sample
    "CRIMS":  24,   # 12/device; binding: vae-3d, ~0.86 GB/sample
}.items()}

# Seeds for the multi-seed grid. Same-seed nondeterminism was measured at
# 0.0005-0.0036 rad SAM — the size of several differences this ablation reports —
# so a single run cannot distinguish a result from noise.
SEEDS = [42, 7, 1234]

HYPERPARAMS = {
    "IIRS": {
        "epochs": 30, "lr": 1e-3, "batch_size": BATCH_SIZE["IIRS"], "num_workers": 4,
        "beta": 1e-3, "lambda_physics": 0.3, "seed": 42,
        "weight_decay": 1e-5, "early_stopping_patience": 7,
        # latent budget: common T = 16,384 (64.0:1). std/3d/1d land EXACTLY;
        # vae-our is +1.6%, entirely its extra 256-dim global vector.
        "spectral_latent_dim": 4,
        "vae_standard_latent_ch": 256,
        "vae_3d_latent_ch": 8,
        "vae_1d_latent_dim": 4,
        # parameters: matched to vae-our = 10,871,625
        "vae_standard_base_ch": 86,
        "vae_3d_base_ch": 45,
        "vae_1d_hidden_dims": (2748, 1374, 687),
    },
    "M3": {
        "epochs": 30, "lr": 1e-3, "batch_size": BATCH_SIZE["M3"], "num_workers": 4,
        "beta": 1e-3, "lambda_physics": 0.3, "seed": 42,
        "weight_decay": 1e-5, "early_stopping_patience": 7,
        # latent budget: common T = 4,096 (84.0:1). std/1d EXACT,
        # vae-3d +3.1%, vae-our +6.2%.
        "spectral_latent_dim": 1,
        "vae_standard_latent_ch": 64,
        "vae_3d_latent_ch": 6,
        "vae_1d_latent_dim": 1,
        # parameters: matched to vae-our = 10,695,435
        "vae_standard_base_ch": 88,
        "vae_3d_base_ch": 45,
        "vae_1d_hidden_dims": (2856, 1428, 714),
    },
    "AVIRIS": {
        "epochs": 30, "lr": 1e-3, "batch_size": BATCH_SIZE["AVIRIS"], "num_workers": 4,
        "beta": 1e-3, "lambda_physics": 0.3, "seed": 42,
        "weight_decay": 1e-5, "early_stopping_patience": 7,
        # latent budget: common T = 28,672. std/1d EXACT,
        # vae-3d -5.4% (AVIRIS) / +1.8% (CRIMS), vae-our +0.9%.
        "spectral_latent_dim": 7,
        "vae_standard_latent_ch": 448,
        "vae_3d_latent_ch": 8,
        "vae_1d_latent_dim": 7,
        # parameters: matched to vae-our = 11,207,199
        "vae_standard_base_ch": 85,
        "vae_3d_base_ch": 46,
        "vae_1d_hidden_dims": (2668, 1334, 667),
    },
    "CRIMS": {
        "epochs": 30, "lr": 1e-3, "batch_size": BATCH_SIZE["CRIMS"], "num_workers": 4,
        "beta": 1e-3, "lambda_physics": 0.3, "seed": 42,
        "weight_decay": 1e-5, "early_stopping_patience": 7,
        # latent budget: common T = 28,672. std/1d EXACT,
        # vae-3d -5.4% (AVIRIS) / +1.8% (CRIMS), vae-our +0.9%.
        "spectral_latent_dim": 7,
        "vae_standard_latent_ch": 448,
        "vae_3d_latent_ch": 8,
        "vae_1d_latent_dim": 7,
        # parameters: matched to vae-our = 11,276,895
        "vae_standard_base_ch": 85,
        "vae_3d_base_ch": 46,
        "vae_1d_hidden_dims": (2656, 1328, 664),
    },
}

_HP_SETTINGS_FIELDS = {
    "batch_size", "num_workers", "epochs", "lr", "beta", "lambda_physics",
    "vae_standard_base_ch", "vae_standard_n_down", "vae_standard_latent_ch",
    "vae_3d_base_ch", "vae_3d_n_down", "vae_3d_latent_ch",
    "vae_1d_hidden_dims", "vae_1d_latent_dim",
    "latent_dim", "spectral_latent_dim",
}


def make_settings(dataset):
    """Settings whose band count — and every derived dim — matches the dataset."""
    return Settings(input_channels=DATASETS[dataset]["input_channels"])


def load_hyperparams(dataset):
    """Per-dataset hyper-parameters (inlined above; no file access)."""
    return dict(HYPERPARAMS[dataset])


def apply_hyperparams(settings, hp):
    """Mutate `settings` in place for the whitelisted fields present in `hp`."""
    for key in _HP_SETTINGS_FIELDS & set(hp):
        value = hp[key]
        if key == "vae_1d_hidden_dims" and isinstance(value, list):
            value = tuple(value)
        setattr(settings, key, value)
    settings.__post_init__()   # recompute derived dims after any knob change


# Global the branch classes read from. Reassigned per dataset in the train loop.
settings = make_settings("IIRS")
print("packed data root:", _PACKED_BASE, "| ckpt root:", CKPT_ROOT)

## **Imports**

In [ ]:
import math
import random
import time
from pathlib import Path
from typing import List

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch import Tensor
from torch.nn import Conv2d, ConvTranspose2d
from torch.utils.data import Dataset, DataLoader

print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "| devices", torch.cuda.device_count())

## **Loss (SAM physics prior + KL)**

In [ ]:
def spectral_angle_mapper_loss(y_true, y_pred):
    """Differentiable physics prior: mean spectral angle (radians)."""
    dot = torch.sum(y_true * y_pred, dim=-1)
    nt = torch.sqrt(torch.sum(y_true ** 2, dim=-1) + 1e-8)
    npd = torch.sqrt(torch.sum(y_pred ** 2, dim=-1) + 1e-8)
    cos = torch.clamp(dot / (nt * npd + 1e-8), -1.0 + 1e-8, 1.0 - 1e-8)
    return torch.mean(torch.acos(cos))

In [ ]:
def kl_divergence(mu, logvar):
    return -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())

## **Metrics (PSNR / SSIM)**

In [ ]:
def compute_psnr(img1, img2, data_range=1.0):
    mse = F.mse_loss(img1, img2)
    if mse == 0:
        return float("inf")
    return (20 * torch.log10(torch.tensor(data_range).to(img1.device)) - 10 * torch.log10(mse)).item()

In [ ]:
def compute_ssim(img1, img2, data_range=1.0, window_size=11):
    if img1.dim() == 4 and img1.shape[-1] not in [img1.shape[1], img1.shape[2]]:
        img1 = img1.permute(0, 3, 1, 2); img2 = img2.permute(0, 3, 1, 2)
    channels = img1.shape[1]

    def gaussian(w, sigma):
        g = torch.exp(torch.tensor([-(x - w // 2) ** 2 / (2 * sigma ** 2) for x in range(w)]))
        return g / g.sum()

    _1d = gaussian(window_size, 1.5).unsqueeze(1).to(img1.device)
    _2d = _1d.mm(_1d.t()).float().unsqueeze(0).unsqueeze(0)
    window = _2d.expand(channels, 1, window_size, window_size).contiguous()
    p = window_size // 2
    c1 = (0.01 * data_range) ** 2; c2 = (0.03 * data_range) ** 2

    # CHUNKED over the batch. Nine (B, C, 64, 64) fp32 temporaries are live at
    # the peak; at B=24, C=456 that is ~1.6 GB on top of the model's own
    # activations, on a 15 GB T4 that is already ~12 GB used. Chunking bounds it
    # and does not change the result (SSIM is averaged over samples).
    total, n = 0.0, 0
    for i in range(0, img1.shape[0], 4):
        a, b = img1[i:i + 4], img2[i:i + 4]
        mu1 = F.conv2d(a, window, padding=p, groups=channels)
        mu2 = F.conv2d(b, window, padding=p, groups=channels)
        mu1_sq, mu2_sq, mu1_mu2 = mu1.pow(2), mu2.pow(2), mu1 * mu2
        s1 = F.conv2d(a * a, window, padding=p, groups=channels) - mu1_sq
        s2 = F.conv2d(b * b, window, padding=p, groups=channels) - mu2_sq
        s12 = F.conv2d(a * b, window, padding=p, groups=channels) - mu1_mu2
        m = ((2 * mu1_mu2 + c1) * (2 * s12 + c2)) / ((mu1_sq + mu2_sq + c1) * (s1 + s2 + c2))
        total += m.mean().item() * a.shape[0]
        n += a.shape[0]
    return total / max(n, 1)


## **Data loader**

In [ ]:
# Packed fp16 shards written by utils/dataset/pack.py: one file per split
# instead of ~15,000 individual .npy patches, already max-normalised and
# already band-cropped. Reading the old layout was the actual bottleneck —
# vae-standard needs 40x fewer FLOPs than vae-our yet took the same 40 min,
# because one epoch opened ~17,700 files totalling ~74 GB.
class PackedPatchDataset(Dataset):
    """Rows of <packed_root>/<split>.npy — (N, H, W, C) float16."""

    def __init__(self, packed_root, split, cache_ram=False):
        path = Path(packed_root) / f"{split}.npy"
        if not path.is_file():
            raise FileNotFoundError(
                f"No packed shard at {path}. Build it with\n"
                f"    PYTHONPATH=. python utils/dataset/pack.py\n"
                f"or point DATA_ROOTS at the unzipped dataset.zip."
            )
        # mmap by default: Kaggle's disk is fast and RAM is only ~13-30 GB.
        self.data = np.load(path, mmap_mode=None if cache_ram else "r")

    def __len__(self):
        return self.data.shape[0]

    def __getitem__(self, idx):
        # Cast to float32 here: leaving the batch in fp16 would put an fp16
        # target against a bf16/fp16 reconstruction inside mse_loss, which is a
        # silent-precision trap for no bandwidth gain (the read is already done).
        return torch.from_numpy(np.asarray(self.data[idx], dtype=np.float32))


def build_dataloader(packed_root, split, batch_size=None, shuffle=True,
                     num_workers=None, pin_memory=True, cache_ram=False):
    ds = PackedPatchDataset(packed_root, split, cache_ram=cache_ram)
    nw = num_workers if num_workers is not None else settings.num_workers
    kwargs = dict(
        batch_size=batch_size or settings.batch_size,
        shuffle=shuffle,
        num_workers=nw,
        pin_memory=pin_memory,
        drop_last=(split == "train"),
    )
    if nw > 0:
        kwargs["persistent_workers"] = True
        kwargs["prefetch_factor"] = 4
    return DataLoader(ds, **kwargs)

## **Model definition**

MODEL DEFINITION  --  vae-1d-pixelwise  (Baseline C: 1D Pixel-Wise VAE)
Per-pixel MLP VAE (Su et al., 2019, deep-autoencoder unmixing): the (B,H,W)
grid is folded entirely into the batch dim, so each pixel spectrum is encoded
independently. Excellent chemistry (low SAM) but no spatial context to denoise
corrupted pixels (poor PSNR/SSIM).

In [ ]:
class VAE_1D_Pixelwise(nn.Module):
    def __init__(self):
        super().__init__()
        c = settings.input_channels
        hidden_dims = tuple(settings.vae_1d_hidden_dims)
        latent_dim = settings.vae_1d_latent_dim
        self.latent_dim = latent_dim
        enc, in_f = [], c
        for h in hidden_dims:
            enc += [nn.Linear(in_f, h), nn.ReLU()]; in_f = h
        enc.append(nn.Linear(in_f, 2 * latent_dim))
        self.encoder = nn.Sequential(*enc)
        dec, in_f = [], latent_dim
        for h in reversed(hidden_dims):
            dec += [nn.Linear(in_f, h), nn.ReLU()]; in_f = h
        dec.append(nn.Linear(in_f, c))
        self.decoder = nn.Sequential(*dec)

    @staticmethod
    def reparameterize(params):
        mu, logvar = torch.chunk(params, 2, dim=-1)
        logvar = torch.clamp(logvar, -30.0, 20.0)
        std = torch.exp(0.5 * logvar)
        return mu + torch.randn_like(std) * std, mu, logvar

    def forward(self, x):
        b, h, w, c = x.shape
        z, mu, logvar = self.reparameterize(self.encoder(x.reshape(b * h * w, c)))
        recon = torch.sigmoid(self.decoder(z)).reshape(b, h, w, c)
        mu = mu.reshape(b, h, w, self.latent_dim)
        logvar = logvar.reshape(b, h, w, self.latent_dim)
        return recon, mu, logvar

    @torch.no_grad()
    def encode_latents(self, x):
        b, h, w, c = x.shape
        mu, _ = torch.chunk(self.encoder(x.reshape(b * h * w, c)), 2, dim=-1)
        return [mu.reshape(b, h, w, self.latent_dim)]

    @torch.no_grad()
    def decode_latents(self, latents):
        z = latents[0]; b, h, w, _ = z.shape
        recon = torch.sigmoid(self.decoder(z.reshape(b * h * w, self.latent_dim)))
        return recon.reshape(b, h, w, settings.input_channels)


def compute_losses(model, x, beta, lambda_physics, use_physics=False):
    recon, mu, logvar = model(x)
    mse = F.mse_loss(recon, x)
    kld = kl_divergence(mu, logvar)
    sam = spectral_angle_mapper_loss(x, recon)
    loss = mse + beta * kld + (lambda_physics * sam if use_physics else 0.0)
    # `mse_final` is the RECONSTRUCTION MSE, the number that is comparable
    # across every model in the grid. For a single-stream model it equals `mse`;
    # for vae-our it does not. Returned separately so the training loop can
    # checkpoint on a quantity that means the same thing in every cell.
    return loss, mse, mse, sam, kld, recon


def build_model():
    return VAE_1D_Pixelwise()


## **Training loop**

In [ ]:
# =============================================================================
# Training loop — mirrors train/train.py's perf work
# =============================================================================
# Changes vs the original notebook loop, all math-preserving:
#   * AMP autocast. T4 (Kaggle) is SM 7.5 and has NO bfloat16, so it falls back
#     to float16 + GradScaler; Ampere+ (A100/lab) uses bfloat16 with no scaler.
#     If final numbers come from Kaggle, that is a dtype difference from the
#     lab/HPC runs — worth stating in the paper's setup section.
#   * channels_last / channels_last_3d memory format.
#   * Metrics accumulate ON GPU; exactly one .item() sync per epoch instead of
#     six per batch.
#   * SSIM/PSNR moved OFF the per-batch training path. An 11x11 windowed SSIM
#     over (B, 64, 64, 424) every step dominated the notebook's runtime for a
#     number that is only reported once per epoch. They are now computed on the
#     validation set only.
#   * nn.DataParallel support via _LossTermsAdapter (see the driver cell).

class _LossTermsAdapter(nn.Module):
    """Expose compute_losses through forward() so nn.DataParallel can split it.

    DP only scatters/gathers the wrapped module's `forward`. Calling
    `model.loss_terms(...)` on a DP wrapper either fails or — worse, via
    `model.module` — silently runs on a single GPU while the other sits idle.
    Routing the loss through forward() is what makes DP actually parallel.
    Scalars are unsqueezed to length-1 so DP can concatenate one row per GPU;
    the caller takes .mean().
    """

    def __init__(self, inner):
        super().__init__()
        self.inner = inner

    def forward(self, x, beta, lambda_physics, use_physics):
        loss, mse, mse_final, sam, kld, recon = compute_losses(
            self.inner, x, beta, lambda_physics, use_physics)
        # `recon` is already batched along dim 0, so DP's gather concatenates it
        # correctly as-is. Unsqueezing it would give (n_gpus, B_per_gpu, H, W, C)
        # and silently corrupt every metric derived from it.
        return (loss.unsqueeze(0), mse.unsqueeze(0), mse_final.unsqueeze(0),
                sam.unsqueeze(0), kld.unsqueeze(0), recon)


def _unwrap(model):
    """The bare model, regardless of DataParallel / adapter wrapping."""
    if isinstance(model, nn.DataParallel):
        model = model.module
    if isinstance(model, _LossTermsAdapter):
        model = model.inner
    return model


def _pick_amp_dtype(device):
    """bfloat16 on Ampere+ (SM 8.0+); float16 elsewhere (Kaggle's T4 is SM 7.5)."""
    if device.type != "cuda":
        return torch.float32, False
    major, _ = torch.cuda.get_device_capability()
    if major >= 8:
        return torch.bfloat16, False       # no GradScaler needed
    return torch.float16, True             # GradScaler required


def train_vae(model, dataloader, epochs, device, use_physics, lr, beta, lambda_physics,
              val_dataloader=None, ckpt_files=None, ckpt_meta=None,
              weight_decay=1e-5, patience=7, log_every=1):
    """
    `ckpt_files` is a dict {"sam": path, "mse": path} — TWO checkpoints per cell.

    `val_loss` has a different FORM in every cell (vae-our carries a 3-branch
    MSE, `physics` cells carry a SAM term, `standard` cells do not), so selecting
    on it meant each cell's weights were chosen by a different objective.
    Selecting everything on SAM breaks the other way: `standard` cells never
    train a SAM term. Writing both lets each analysis read the checkpoint
    selected on the metric it reports, uniformly across all cells.
    """
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    model.to(device)
    ckpt_meta = ckpt_meta or {}
    ckpt_files = ckpt_files or {}
    best = {k: math.inf for k in ckpt_files}
    best_epoch = {k: 0 for k in ckpt_files}
    no_improve = 0
    for _p in ckpt_files.values():
        Path(_p).parent.mkdir(parents=True, exist_ok=True)

    amp_dtype, needs_scaler = _pick_amp_dtype(device)
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=needs_scaler)

    if device.type == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        torch.backends.cudnn.benchmark = True

    for epoch in range(1, epochs + 1):
        t0 = time.time()
        model.train()
        # On-GPU accumulators: no host sync inside the loop.
        acc = torch.zeros(4, device=device)
        for x in dataloader:
            x = x.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=use_amp, dtype=amp_dtype):
                loss, mse, _mf, sam, kld, _ = model(x, beta, lambda_physics, use_physics)
                loss = loss.mean()          # DP returns one row per GPU
            if needs_scaler:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
            with torch.no_grad():
                acc += torch.stack([loss.detach(), mse.detach().mean(),
                                    sam.detach().mean(), kld.detach().mean()])
        n = max(len(dataloader), 1)
        tl, tm, ts, tk = (acc / n).tolist()      # one sync per epoch

        vl = vm = vs = vk = 0.0
        vmf = vp = vss = 0.0
        if val_dataloader is not None:
            model.eval()
            vacc = torch.zeros(5, device=device)
            vp_sum = vss_sum = 0.0
            with torch.no_grad(), torch.amp.autocast("cuda", enabled=use_amp, dtype=amp_dtype):
                for j, x in enumerate(val_dataloader):
                    x = x.to(device, non_blocking=True)
                    loss, mse, mse_final, sam, kld, recon = model(
                        x, beta, lambda_physics, use_physics)
                    vacc += torch.stack([loss.mean(), mse.mean(), mse_final.mean(),
                                         sam.mean(), kld.mean()])
                    # PSNR/SSIM only here, and only in float32 — the windowed
                    # conv in compute_ssim is numerically touchy under fp16.
                    vp_sum += compute_psnr(x.float(), recon.float())
                    vss_sum += compute_ssim(x.float(), recon.float())
            nv = max(len(val_dataloader), 1)
            vl, vm, vmf, vs, vk = (vacc / nv).tolist()
            vp, vss = vp_sum / nv, vss_sum / nv
        scheduler.step()

        if epoch % log_every == 0 or epoch == epochs:
            print(f"Epoch [{epoch}/{epochs}] {time.time() - t0:6.1f}s  "
                  f"Loss {tl:.4f} MSE {tm:.4f} SAM {ts:.4f} KLD {tk:.4f}"
                  + (f" | Val Loss {vl:.4f} MSE {vm:.4f} MSEf {vmf:.4f} "
                     f"SAM {vs:.4f} PSNR {vp:.2f} SSIM {vss:.4f}"
                     if val_dataloader else "")
                  + (f" | peak {torch.cuda.max_memory_allocated() / 2**30:.1f} GB"
                     if device.type == "cuda" else ""))

        # Both criteria are minimised, each into its own file.
        monitors = ({"sam": vs, "mse": vmf} if val_dataloader is not None
                    else {"sam": ts, "mse": tm})
        improved_any = False
        for crit, path in ckpt_files.items():
            if monitors[crit] >= best[crit]:
                continue
            best[crit] = monitors[crit]
            best_epoch[crit] = epoch
            improved_any = True
            # Save the UNWRAPPED state dict so the checkpoint loads without
            # DataParallel's "module." prefix on every key.
            torch.save({"epoch": epoch,
                        "model_state_dict": _unwrap(model).state_dict(),
                        "optimizer_state_dict": optimizer.state_dict(),
                        "loss": monitors[crit], "select": crit,
                        "val_sam": vs, "val_mse_final": vmf,
                        "val_psnr": vp, "val_ssim": vss, **ckpt_meta}, path)
        no_improve = 0 if improved_any else no_improve + 1

        # Stop only when NEITHER criterion has improved; stopping on one alone
        # would truncate the other's search.
        if val_dataloader is not None and no_improve >= patience:
            print(f"Early stopping at epoch {epoch}: neither val SAM nor val "
                  f"recon-MSE improved for {patience} epochs.")
            break

    for crit, path in ckpt_files.items():
        print(f"  best-{crit}: epoch {best_epoch[crit]} ({best[crit]:.6f}) -> {path}")
    return best

## **Train across all datasets × loss regimes**

In [ ]:
# =============================================================================
# Train across the grid
# =============================================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_GPUS = torch.cuda.device_count() if device.type == "cuda" else 0
# Kaggle hands out 2x T4. DataParallel splits the batch across them, so the
# per-device load is batch_size/2. The per-dataset batch sizes are derived
# against the lab's 24 GB, and every one of them clears a 15 GB T4 at half
# load (worst case CRIMS: 8 x 0.961 + 0.22 ~= 7.9 GB).
USE_DP = N_GPUS > 1
print(f"device: {device} | GPUs: {N_GPUS}"
      + (f" ({torch.cuda.get_device_name(0)})" if N_GPUS else "")
      + (f" | nn.DataParallel ON, batch split across {N_GPUS}" if USE_DP else ""))

MODEL_NAME = "vae-1d-pixelwise"
LOSS_TYPES = ["standard", "physics"]

# =============================================================================
# WHAT THIS SESSION WILL RUN  -- read before starting, Kaggle kills at ~12 h
# =============================================================================
# The full sweep (4 datasets x 3 seeds) does NOT fit in one Kaggle session. On
# the lab's card the vae-our sweep alone measured ~10 h at 30 epochs; a T4 is
# 3-4x slower and DataParallel across two of them recovers well under half of
# that, so the full sweep is ~20 h+. It will be killed part-way.
#
# So: run ONE dataset (or one seed) per session and change these two lines
# between sessions. Completed cells are SKIPPED on restart (see SKIP_EXISTING),
# so re-running after a timeout resumes instead of starting over -- but only if
# CKPT_ROOT survives, i.e. save /kaggle/working as a dataset or output between
# sessions.
#
# CLAIM cells (the physics regime, where the ablation's claim lives) get every
# seed; `standard` cells get the first seed only, because the extra GPU time
# would otherwise go to cells no claim depends on.
RUN_DATASETS = ["IIRS"]          # <-- ONE dataset per session
RUN_SEEDS = SEEDS                # <-- or narrow to e.g. [42] and vary datasets

# Re-running a session skips cells whose BOTH checkpoints already exist. Set to
# False to force a full retrain.
SKIP_EXISTING = True

for ds in RUN_DATASETS:
    for loss_type in LOSS_TYPES:
        seeds_here = RUN_SEEDS if loss_type == "physics" else RUN_SEEDS[:1]
        for seed in seeds_here:
            print("\n" + "=" * 60)
            print(f" TRAIN {MODEL_NAME} | {ds} | {loss_type} | seed {seed}")
            print("=" * 60)

            settings = make_settings(ds)
            hp = load_hyperparams(ds)
            apply_hyperparams(settings, hp)
            globals()["settings"] = settings   # branch classes read the module global
            # The seed must reach the CHECKPOINT NAME too, or two seeds overwrite
            # each other and the whole exercise measures nothing.
            torch.manual_seed(seed)
            random.seed(seed)
            np.random.seed(seed)
            patience = hp.get("early_stopping_patience", 7)
            weight_decay = hp.get("weight_decay", 1e-5)

            try:
                train_loader = build_dataloader(DATA_ROOTS[ds], "train", shuffle=True)
                val_loader   = build_dataloader(DATA_ROOTS[ds], "valid", shuffle=False)
            except FileNotFoundError as e:
                # Don't kill the whole sweep because one dataset wasn't uploaded.
                print(f"  !! skipping {ds}: {e}")
                continue
            print(f"  C={settings.input_channels} | train batches {len(train_loader)} "
                  f"| val batches {len(val_loader)} | batch {settings.batch_size}")

            raw_model = build_model().to(device)
            with torch.no_grad():   # materialize any Lazy layers before wrapping
                raw_model(torch.randn(2, settings.input_height, settings.input_width,
                                      settings.input_channels, device=device))
            n_params = sum(p.numel() for p in raw_model.parameters())
            print(f"  params: {n_params:,} ({n_params / 1e6:.2f}M)")

            # channels_last lets cuDNN pick its faster kernels on Ampere+; it is a
            # harmless no-op where the convs don't benefit.
            mem_fmt = (torch.channels_last_3d if MODEL_NAME == "vae-3d-spatio-spectral"
                       else torch.channels_last)
            try:
                raw_model = raw_model.to(memory_format=mem_fmt)
            except Exception:
                pass

            # Wrap for the loss AFTER materializing, so DP scatters the real work.
            model = _LossTermsAdapter(raw_model)
            if USE_DP:
                model = nn.DataParallel(model)
            model.to(device)

            stem = f"{MODEL_NAME}_{loss_type}_seed{seed}"
            ckpt_files = {c: Path(CKPT_ROOT) / ds / f"{stem}_best{c}.pt"
                          for c in ("sam", "mse")}
            if SKIP_EXISTING and all(f.is_file() for f in ckpt_files.values()):
                print(f"  [skip] both checkpoints already exist for seed {seed}")
                continue
            train_vae(model, train_loader, settings.epochs, device,
                      use_physics=(loss_type == "physics"),
                      lr=settings.lr, beta=settings.beta,
                      lambda_physics=settings.lambda_physics,
                      val_dataloader=val_loader, ckpt_files=ckpt_files,
                      ckpt_meta={"model": MODEL_NAME, "dataset": ds,
                                 "loss_type": loss_type, "seed": seed,
                                 # batch size differs between this notebook (Kaggle
                                 # 30 GB) and the scripts (lab 20 GB); record it so
                                 # the confound is auditable from the artifact.
                                 "batch_size": settings.batch_size,
                                 "platform": "kaggle-notebook"},
                      weight_decay=weight_decay, patience=patience)